# はじめに

このノートではPythonのpolarsライブラリの使い方をまとめておく。polarsライブラリはRustで書かれており、高速に動作することやメモリ効率が良いことが特徴での大規模データを処理するのに適しているライブラリ。

- [Polars — DataFrames for the new era](https://pola.rs/)

## ライブラリの読み込み

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.9.1
False


## パイプ処理

polarsライブラリの関数はR言語のmagrittrパッケージが提供するパイプ演算子`|>`を使ったパイプライン処理が可能。ただ、パイプを繋げる際は、バックスラッシュ`\`をつけて改行する必要があるので注意が必要。



In [36]:
import torch
import polars as pl
torch.manual_seed(0)

# -------------------------
# ダミー競艇データ
# -------------------------


def make_race_data(n_races):
    """
    各レース:
      X: (6, 3)  特徴量
      y: (6,)    正解順位（1が1着）
    """
    data = []
    for _ in range(n_races):
        X = torch.randn(6, 3)

        # 真の強さ（ダミー）
        true_strength = (
            1.5 * X[:, 0]
            - 1.0 * X[:, 1]
            + 0.5 * X[:, 2]
            + 0.1 * torch.randn(6)
        )

        # rank: 1 強い
        _, order = torch.sort(true_strength, descending=True)
        rank = torch.empty_like(order)
        rank[order] = torch.arange(1, 7)

        data.append((X, rank))
    return data


train_data = make_race_data(50)
test_data = make_race_data(10)
pred_data = make_race_data(1)
train_data[:3]
# X, rank = train_data[0]
# df = pl.DataFrame(
#     torch.column_stack([X, rank]).numpy(),
#     schema=["feature1", "feature2", "feature3", "rank"]
# )
# df

[(tensor([[-1.1258, -1.1524,  0.5667],
          [ 0.7935,  0.5988, -1.5551],
          [-0.3414,  1.8530,  0.4681],
          [-0.1577, -0.1734,  0.1835],
          [ 1.3894,  1.5863,  0.9463],
          [-0.8437,  0.9318,  1.2590]]),
  tensor([4, 3, 6, 2, 1, 5])),
 (tensor([[ 0.9383,  0.4889, -0.5692],
          [ 0.9200,  1.1108,  1.2899],
          [-1.4782,  2.5672, -0.4731],
          [ 0.3356, -1.6293, -0.5497],
          [-0.4798, -0.4997, -1.0670],
          [ 1.1149, -0.1407,  0.8058]]),
  tensor([4, 3, 6, 2, 5, 1])),
 (tensor([[ 1.9415,  0.7915, -0.7502],
          [-1.3120, -0.2188, -2.4351],
          [-0.0729, -0.0340,  0.9625],
          [ 0.3492, -0.3701, -1.2103],
          [-0.6227, -0.4637,  1.9218],
          [-0.4025,  0.1239,  1.1648]]),
  tensor([1, 6, 3, 4, 2, 5]))]

In [ ]:
import torch.nn as nn


class BoatRankNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 1)

    def forward(self, x):
        # x: (6, 3)
        return self.linear(x).squeeze(-1)  # (6,)


def ranknet_loss(scores, ranks):
    """
    scores: (6,)
    ranks:  (6,)  1が1着
    """
    loss = 0.0
    n = scores.size(0)

    for i in range(n):
        for j in range(n):
            if ranks[i] < ranks[j]:  # iの方が上位
                loss += torch.log1p(torch.exp(-(scores[i] - scores[j])))
    return loss

In [37]:
model = BoatRankNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -------------------------
# 学習
# -------------------------
for epoch in range(30):
    total_loss = 0.0
    for X, rank in train_data:
        optimizer.zero_grad()
        scores = model(X)
        loss = ranknet_loss(scores, rank)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 5 == 0:
        print(f"epoch {epoch}, loss={total_loss:.2f}")

epoch 0, loss=394.57
epoch 5, loss=315.90
epoch 10, loss=267.14
epoch 15, loss=233.68
epoch 20, loss=209.00
epoch 25, loss=189.85


In [38]:
# -------------------------
# テスト
# -------------------------
def soft_ndcg(scores, ranks):
    """
    SoftNDCG: 予測スコアと真のランクからNDCGを計算
    scores: (6,) 予測スコア
    ranks:  (6,) 真のランク（1が1着）
    """
    # 関連度: ランクが小さいほど関連度が高い（1着が最も関連度が高い）
    relevance = 1.0 / ranks.float()  # または 7 - ranks.float() など

    # 予測順序でソート
    _, pred_order = torch.sort(scores, descending=True)
    pred_relevance = relevance[pred_order]

    # DCG計算
    dcg = 0.0
    for i in range(len(pred_relevance)):
        dcg += pred_relevance[i] / torch.log2(torch.tensor(i + 2.0, dtype=torch.float32))

    # IDCG計算（真のランク順に並べた場合）
    _, true_order = torch.sort(ranks)
    ideal_relevance = relevance[true_order]
    idcg = 0.0
    for i in range(len(ideal_relevance)):
        idcg += ideal_relevance[i] / torch.log2(torch.tensor(i + 2.0, dtype=torch.float32))

    # NDCG
    if idcg == 0:
        return 0.0
    return (dcg / idcg).item()


print("\n--- test ---")
for X, rank in test_data[:3]:
    scores = model(X)
    print("score:", scores.detach().numpy())
    print("true :", rank.numpy())
    print("SoftNDCG:", soft_ndcg(scores, rank))


--- test ---
score: [ 0.5952207  0.8192737 -1.9353142  1.1765964  0.863788   1.3976065]
true : [4 3 6 2 5 1]
SoftNDCG: 0.9933773875236511
score: [-0.6319046  -0.99622023 -3.347727    0.8866004   1.8324094   0.4309862 ]
true : [4 5 6 2 1 3]
SoftNDCG: 1.0
score: [ 2.5112674  1.54121    2.2371366 -3.207317  -0.5973258  0.9195355]
true : [2 3 1 6 5 4]
SoftNDCG: 0.8931184411048889


In [ ]:
# -------------------------
# 予測
# -------------------------
X_pred, _ = pred_data[0]

scores = model(X_pred)
order = torch.argsort(scores, descending=True)

print("\n--- prediction ---")
print("scores:", scores.detach().numpy())
print("pred rank (1=1着):", order.numpy()+1)


--- prediction ---
scores: [-9.9328575  -0.03673881 12.035468   -4.4013915  -8.292181   -8.24094   ]
pred rank (1=1着): [3 2 4 6 5 1]
